In [2]:
# Code for working in BlueBEAR (UoB HPC)
# Set of commands to run every time to work from the virtual environment

import os
from pathlib import Path
import sys
import site
node_type = os.getenv('BB_CPU')
venv_dir = f'my-virtual-env-{node_type}'  # edit this line to match the venv directory format
venv_site_pkgs = Path(venv_dir) / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
if venv_site_pkgs.exists():
    site.addsitedir(str(venv_site_pkgs))
else:
    print(f"Path '{venv_site_pkgs}' not found. Check that it exists and/or that it exists for node-type '{node_type}'.")

# Basically stores the filepath for the virtual environment packages
venv_site_pkgs = "/rds/projects/b/bravol-tuberculosis/lewis-code/TB-scRNAseq/my-virtual-env-emerald/lib/python3.12/site-packages"
# Then sets this path as the first file path to go through when importing packages
sys.path.insert(0, venv_site_pkgs)

In [6]:
import scanpy as sc
import scipy
from scipy.sparse import save_npz, load_npz
import pandas as pd
import anndata as ad

In [14]:
# Do not load this in the future unless you don't have the new h5ad file generated
# in a few lines
adata = sc.read_h5ad('GSE326212_human_tb_bal_adata.h5ad')

In [4]:

# Assign the counts matrix to a variable
counts = adata.layers['counts']
# And save to a .npz file so that i can construct a new h5ad file of just counts
# , gene names, and cell identifiers
save_npz("counts.npz", counts)

In [ ]:
# Save gene names
pd.Series(adata.var_names).to_csv("gene_names.csv", index=False)

# Save cell identifiers
pd.Series(adata.obs_names).to_csv("cell_names.csv", index=False)

In [7]:
# Load all the counts
counts = load_npz("counts.npz")

FileNotFoundError: [Errno 2] No such file or directory: 'cell_metadata.csv'

In [8]:
# Load the csv files
obs = pd.read_csv("cell_names.csv", index_col=0)
var = pd.read_csv("gene_names.csv", index_col=0)

In [9]:
# Create an anndata object
adata = ad.AnnData(
    X=counts,
    obs=obs,
    var=var
)

In [10]:
# Write this to a file
adata.write_h5ad("raw_counts.h5ad")

## Prep

In [ ]:
# Filter out genes that are present in only 10 cells
#sc.pp.filter_genes(adata, min_cells=10)

In [ ]:
adata

In [ ]:
# Only keep the 2000 top variable genes
# So 2000 genes that describe the data best
sc.pp.highly_variable_genes(adata,
                            n_top_genes = 2000,
                            subset = True,
                            flavor= "seurat_v3")